# Train & evaluate the MoE toxicity classifier on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qpp112/toxic-comment-moe/blob/main/notebooks/train_on_colab.ipynb)

This notebook reproduces every number in the [README](https://github.com/qpp112/toxic-comment-moe):

1. checks the GPU and installs the package,
2. runs the test suite (1 min, catches problems before any GPU time is spent),
3. loads the Jigsaw data,
4. trains the **loss × head ablation**: 8 settings × 3 seeds = 24 runs on BERT-base,
5. trains the **stronger-encoder study**: 2 settings × 3 seeds = 6 runs on DeBERTa-v3-base,
6. builds the results tables and figures, and lets you try the model.

**GPU:** *Runtime → Change runtime type* → **A100** (or **L4** if no A100 is available). Both
run in bf16. A T4 works for the BERT runs (fp16) but is several times slower, and DeBERTa on a
T4 falls back to fp32.

**Time:** the runner prints the real pace after the first run. As a rough guide, on an A100
a BERT-base run should take a few minutes and a DeBERTa run about twice that. Results go to
Google Drive and finished runs are skipped, so **if Colab disconnects, re-run the setup cells
and the study cell** and it continues where it stopped.

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    bf16 = torch.cuda.get_device_capability(0)[0] >= 8
    print("GPU:", name, "| bf16:", bf16)
else:
    print("No GPU found. Runtime -> Change runtime type -> A100 (or L4), then re-run this cell.")

## 1. Settings, Google Drive, install

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/qpp112/toxic-comment-moe.git"
USE_DRIVE = True  # keep data, tokenization cache and results on Drive so a disconnect loses nothing

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/toxic-moe")
else:
    WORK = Path("/content/toxic-moe-work")

DATA_DIR = WORK / "data"
RESULTS_DIR = WORK / "results"
for d in (DATA_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   ->", DATA_DIR)
print("results->", RESULTS_DIR)

In [ ]:
%cd /content
!test -d toxic-comment-moe || git clone -q {REPO_URL}
%cd /content/toxic-comment-moe
!git pull -q
!pip install -q -e ".[dev,deberta]"

import sys

sys.path.insert(0, "/content/toxic-comment-moe/src")  # editable installs need a runtime restart otherwise

## 2. Run the test suite (~1 min)

In [ ]:
!python -m pytest -q

## 3. Get the data

**Easiest (no API key):**

1. Log in to Kaggle, open
   <https://www.kaggle.com/competitions/jigsaw-toxic-comment-classification-challenge/data>,
   accept the rules if asked, and click **Download All**. You get
   `jigsaw-toxic-comment-classification-challenge.zip` (~55 MB).
2. Upload that zip, unchanged, to Google Drive in **My Drive → toxic-moe → data**. The
   folder was created by the settings cell above.
3. Run the cell below. It unzips everything (including the zips inside the zip).

If the zip is not on Drive, the cell shows an upload button instead. That works too, but
uploading through the browser is slower. Set `USE_KAGGLE_API = True` to download with a
`kaggle.json` key instead.

In [ ]:
from toxic_moe.data import REQUIRED_FILES, ensure_extracted

USE_KAGGLE_API = False


def have_csvs():
    return all((DATA_DIR / f).exists() for f in REQUIRED_FILES)


if not have_csvs() and any(DATA_DIR.glob("*.zip")):
    ensure_extracted(DATA_DIR)

if not have_csvs() and not USE_KAGGLE_API:
    from google.colab import files

    print("Upload jigsaw-toxic-comment-classification-challenge.zip (from the Kaggle 'Download All' button)")
    for name, blob in files.upload().items():
        (DATA_DIR / name).write_bytes(blob)
    ensure_extracted(DATA_DIR)

if not have_csvs() and USE_KAGGLE_API:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        from google.colab import files

        print("Upload kaggle.json (Kaggle -> Settings -> API -> Create Legacy API Key)")
        uploaded = files.upload()
        kaggle_json.parent.mkdir(exist_ok=True)
        kaggle_json.write_bytes(next(iter(uploaded.values())))
        kaggle_json.chmod(0o600)
    !pip install -q kaggle
    !bash scripts/download_data.sh {DATA_DIR}

print("CSV files:", sorted(p.name for p in DATA_DIR.glob("*.csv")))

## 4. Quick pipeline check (~5 min)

One MoE run on 2% of the data for one epoch. This also tokenizes and caches the 64k-comment
test set on Drive, so later runs start faster. The scores here are meaningless; it only checks
that everything runs end to end. Skip this cell when resuming after a disconnect.

In [ ]:
!python scripts/run_ablation.py --only moe_cbfocal --seeds 42 --set train_fraction=0.02 epochs=1 data_dir={DATA_DIR} output_dir=/content/smoke_results --force

## 5. Main study: loss × head ablation, 3 seeds (24 runs)

Runs are ordered seed-major: all 8 settings for seed 42, then 43, then 44. Even a partial
study is a complete single-seed comparison. After each run the script prints how long it took
and an estimate of the time left.

In [ ]:
!python scripts/run_ablation.py --list --set data_dir={DATA_DIR} output_dir={RESULTS_DIR}

In [ ]:
!python scripts/run_ablation.py --set data_dir={DATA_DIR} output_dir={RESULTS_DIR}

## 6. Stronger encoder: DeBERTa-v3-base, 3 seeds (6 runs)

Baseline (linear head + BCE) vs. headline (MoE head + class-balanced focal) with DeBERTa-v3-base
instead of BERT-base. DeBERTa runs in **full precision** (fp32 with fast TF32 matmuls on an
A100/L4), because under bf16 it produced NaN losses. That makes each run slower than a BERT run.

First a 2-minute check on 1% of the data. The logged `loss` must be a number, not `nan`, and the
final `TEST ROC-AUC` should be well above 0.5. If a run ever goes NaN, it now stops by itself
after 20 bad steps instead of wasting the GPU.

In [ ]:
!python scripts/run_ablation.py --config configs/encoders.yaml --only deberta_bce --seeds 42 --set train_fraction=0.01 epochs=1 log_every=20 data_dir={DATA_DIR} output_dir=/content/smoke_results --force

In [ ]:
# Full study. To save compute units, add `--seeds 42 43` to run 2 seeds instead of 3.
!python scripts/run_ablation.py --config configs/encoders.yaml --set data_dir={DATA_DIR} output_dir={RESULTS_DIR}

## 7. Report

In [ ]:
import re

from IPython.display import Image, Markdown, display

!python scripts/make_report.py --results {RESULTS_DIR} --update-readme

summary = (RESULTS_DIR / "summary.md").read_text()
display(Markdown(re.sub(r"<picture>.*?</picture>", "", summary, flags=re.S)))
for stem in ["ablation", "encoders", "per_label_f1", "pr_curves", "expert_routing"]:
    path = Path("docs/figures") / f"{stem}_light.png"
    if path.exists():
        display(Image(str(path)))

### Save the results

Downloads a small zip (a few MB) with the metrics, configs, figures and the updated README.
Checkpoints and predictions are not included. Put the zip in your Resume folder and it gets
committed to the repository.

In [ ]:
import zipfile

bundle = Path("/content/toxic_moe_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file() and p.suffix in {".json", ".yaml", ".md"}:
            z.write(p, Path("results") / p.relative_to(RESULTS_DIR))
    for p in Path("docs/figures").glob("*.png"):
        z.write(p, p)
    z.write("README.md", "README.md")
print(f"{bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")

from google.colab import files

files.download(str(bundle))

## 8. Try the model

Loads the saved `moe_cbfocal_s42` checkpoint (the only run that keeps its weights) and scores
some sentences. `flagged` uses the per-label thresholds tuned on the validation split. The
`E1`–`E6` columns show how the router split each comment across the six experts.

In [ ]:
import pandas as pd

from toxic_moe.predict import ToxicityPredictor

predictor = ToxicityPredictor(RESULTS_DIR / "moe_cbfocal_s42")

examples = [
    "Thanks for fixing the citation, much appreciated!",
    "Please stop reverting my edits without explaining why on the talk page.",
    "You are an idiot and nobody wants you editing here.",
    "If you touch this article again I will find you and make you regret it.",
    "People from that country are all criminals and should be banned.",
    "This is the dumbest, most useless garbage I have ever read.",
]
rows = predictor(examples)
table = pd.DataFrame(
    [
        {
            "text": r["text"][:70],
            **r["probabilities"],
            "flagged": ", ".join(r["flagged"]) or "-",
            **{f"E{i + 1}": w for i, w in enumerate(r["expert_weights"])},
        }
        for r in rows
    ]
)
table.style.format(precision=2).background_gradient(cmap="Blues", subset=list(rows[0]["probabilities"]))

In [ ]:
# Your own sentence:
text = "Type something here"
predictor([text])[0]